#Análise de Sentimento com BERT - Bertimbau
Nesse caderno, nós vamos utilizar modelos de classificação baseados em Transformers para implementar uma aplicação de classificação de documentos, mais especificamente para análise de sentimentos. Nós vamos utilizar o modelo BERT, proposto pelo Google, carregando uma versão pré-treinada com a biblioteca [Transformers da Hugging Face](https://huggingface.co/docs/transformers/index).

Nesse caderno, vamos utilizar os seguintes passos de utilização de um modelo pré-treinado baseado em Transformers:
1. Carregamento do dataset (similar à forma como já fizemos em outros exemplos);
2. Carregamento do modelo pré-treinado (exercitando a técnica de Transfer learning);
3. Treinamento do modelo (refinamento do modelo para aplicações específicas);
4. Avaliação do modelo.

Inicialmente, vamos fazer o download do nosso dataset. Vamos utilizar o mesmo dataset que utilizamos em aulas anteriores para análise de sentimentos da plataforma de E-Commerce - Buscapé.

In [ ]:
import requests, os, pandas as pd

from sklearn.model_selection import train_test_split


# download the dataset
def download (url, filename=''):
  if (os.path.isfile(filename)):
    print('Arquivo já existente no Runtime... Tudo OK')
    return
  response = requests.get(url)
  with open(f'./{filename}', 'wb') as f:
      f.write(response.content)
      print('Download realizado e arquivo extraído no Runtime... Tudo OK')


url = "https://raw.githubusercontent.com/watinha/nlp-text-mining-datasets/main/buscape.csv"
download(url, filename='buscape.csv')

df = pd.read_csv('buscape.csv').dropna()
X = df[['review_text', 'original_index']]
y = df['polarity'].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

X_train = X_train['review_text'].to_numpy()
X_test = X_test['review_text'].to_numpy()

print(f'X_train shape {X_train.shape}')
print(f'X_test shape {X_test.shape}')



Download realizado e arquivo extraído no Runtime... Tudo OK
X_train shape (55219,)
X_test shape (18407,)


## Carregamento do modelo pré-treinado

Após carregar o dataset da nossa aplicação, vamos carregar o modelo pré-treinado da nossa proposta. Nesse exemplo, nós vamos utilizar um modelo Bert pré-treinado. Esse modelo pré-treinado é identificado como [Bertimbau](https://huggingface.co/neuralmind/bert-base-portuguese-cased) e é mantido pela empresa brasileira [NeuralMind](https://neuralmind.ai/en/home-en/).

Para utilizar modelos pré-treinados de NLP e, posteriormente, refinar esses modelos com treinamentos com datasets específicos, a bibliote Transformers utiliza-se de dois componentes: o Tokenizer que gera sequências de tokens e máscaras de atenção; e o modelo BERT.

O Tokenizer tem como objetivo gerar sequências de forma similar às APIs de TextVectorization que utilizamos anteriormente. No entanto, vamos utilizar um Tokenizer com o vocabulário já definido/treinado. Esse componente Tokenizer tem como objetivo gerar sequências e máscaras de atenção que serão dadas como entrada para o modelo BERT.

Para carregar o Tokenizer pré-treinado, vamos utilizar o comando from_pretrained identificando o modelo que desejamos carregar. Nesse caso, identificamos o modelo pré-treinado, Bertimbau. Esses modelos são carregados do website da empresa HuggingFace que disponibiliza uma ampla variedade de modelos pré-treinados abertos. O modelo Bertimbau foi selecionado para o exemplo, principalmente considerando a linguagem/idioma para o qual ele foi treinado.

Ao executar o Tokenizer, nós passamos atributos relacionados ao uso de: padding/truncation/max_length para garantir que todas as sequências tenham o mesmo tamanho; e o parâmetro return_tensors com o falor de "tf". Esse último parâmetro serve para identificar os tipos de tensores que serão utilizados. A API Transformers permite que utilizemos modelos com base nas bibliotecas PyTorch e TensorFlow. Nesse caso, estamos explicitamente informando a API para utilizarmos o back-end baseado na biblioteca Tensorflow que utilizamos até agora na disciplina.

In [ ]:
from transformers import AutoTokenizer


tokenizer = AutoTokenizer.from_pretrained("neuralmind/bert-base-portuguese-cased")

X_train_encoded = tokenizer(
    X_train.tolist(), padding=True, truncation=True, max_length=50,
    return_tensors="tf")
X_test_encoded = tokenizer(
    X_test.tolist(), padding=True, truncation=True, max_length=50,
    return_tensors="tf")

print(X_train_encoded.keys())
print(X_train_encoded['input_ids'].shape)
print(X_train_encoded)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/210k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

dict_keys(['input_ids', 'token_type_ids', 'attention_mask'])
(55219, 50)
{'input_ids': <tf.Tensor: shape=(55219, 50), dtype=int32, numpy=
array([[  101, 15945, 22003, ...,   117,  2251,   102],
       [  101, 12925,  4062, ...,     0,     0,     0],
       [  101,  1643,  8705, ..., 22283,   179,   102],
       ...,
       [  101,   231, 12682, ...,   202, 17105,   102],
       [  101,  4062,   125, ...,     0,     0,     0],
       [  101,  9235,  3576, ...,     0,     0,     0]], dtype=int32)>, 'token_type_ids': <tf.Tensor: shape=(55219, 50), dtype=int32, numpy=
array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], dtype=int32)>, 'attention_mask': <tf.Tensor: shape=(55219, 50), dtype=int32, numpy=
array([[1, 1, 1, ..., 1, 1, 1],
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 1, 1, 1],
       ...,
       [1, 1, 1, ..., 1, 1, 1],


Após o carregamento do tokenizer, nós podemos verificar os resultados gerados ao se utilizar o componente. Observem que ele gera sequências de tokens codificados em valores inteiros e máscaras de atenção para identificar temos que devem ser analisados juntos nos modelos de linguagem.

Em seguida, nós podemos carregar o modelo Bert pré-treinado para realizar a classificação de documentos. Observem que utilizamos a classe TFBertForSequenceClassification. Essa classe foi utilizada considerando o tipo de aplicação que estamos desenvolvendo (Classificação de Sequências) e o back-end dos modelos que estamos utilizando, o TF para TensorFlow.

In [ ]:
from transformers import TFBertForSequenceClassification, AdamWeightDecay
from sklearn.metrics import classification_report


model = TFBertForSequenceClassification.from_pretrained(
    "neuralmind/bert-base-portuguese-cased")
model.compile(optimizer=AdamWeightDecay(learning_rate=2e-5), metrics=['accuracy'])
model.fit(X_train_encoded, y_train, epochs=5, validation_split=0.1)



tf_model.h5:   0%|          | 0.00/529M [00:00<?, ?B/s]

All model checkpoint layers were used when initializing TFBertForSequenceClassification.

Some layers of TFBertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier', 'bert/pooler/dense/bias:0', 'bert/pooler/dense/kernel:0']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/5


Cause: for/else statement not yet supported
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Cause: for/else statement not yet supported
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
1554/1554 [==============================] - 229s 105ms/step - loss: 0.1559 - accuracy: 0.9504 - val_loss: 0.1461 - val_accuracy: 0.9560
Epoch 2/5
1554/1554 [==============================] - 152s 98ms/step - loss: 0.1156 - accuracy: 0.9653 - val_loss: 0.1418 - val_accuracy: 0.9558
Epoch 3/5
1554/1554 [==============================] - 152s 98ms/step - loss: 0.0858 - accuracy: 0.9764 - val_loss: 0.1688 - val_accuracy: 0.9545
Epoch 4/5
1554/1554 [==============================] - 152s 98ms/step - loss: 0.0599 - accuracy: 0.9842 - val_loss: 0.1770 - val_accuracy: 0.9511
Epoch 5/5
1554/1554 [==============================] - 152s 98ms/step - loss: 0.0461 - accuracy: 0.9878 - val_loss: 0.2000 - val_accuracy: 0.9524


## Treinamento/Refinamento e Avaliação
Após realizarmos o carregamento do modelo, nós podemos refina-lo, realizando treinamentos com dados específicos relacionados à aplicação que está sendo implementada.

Por fim, nós podemos utilizar esse modelo para predizer o sentimento de postagens e analisar as métricas de acurácia do modelo gerado.

In [ ]:
tf_output = model.predict(X_test_encoded)
y_pred = tf_output.logits.argmax(axis=-1)
print(classification_report(y_test, y_pred))

576/576 [==============================] - 30s 38ms/step
              precision    recall  f1-score   support

         0.0       0.81      0.62      0.70      1705
         1.0       0.96      0.99      0.97     16702

    accuracy                           0.95     18407
   macro avg       0.89      0.80      0.84     18407
weighted avg       0.95      0.95      0.95     18407

